# Exploratory analysis — Siddipet maize

Reshapes Earth Engine zonal-stats output into growth-stage vegetation indices, draws a correlation heatmap, and compares Linear Regression with Random Forest against CCE plot weight.

**Inputs:** `Siddipet_cce.csv` (from `maize_yield_siddipet_thoothukudi`)  
**Outputs:** `Predicted_revised.xlsx`  

> Update the path variables in the first cells before running. See [`docs/`](../../docs/) for methodology and parameters.
>
> ⚠️ Uses `criterion='mse'`, which needs scikit-learn < 1.2. Change it to `'squared_error'` for newer versions.

In [ ]:
import pandas as pd
import os

In [ ]:
import numpy as np
import seaborn as sns

In [ ]:
data=pd.read_csv(r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\raw\Siddipet_Maize_Rabi2022-23\shape\Siddipet_cce.csv")

In [ ]:
data.columns

In [ ]:
data_map=data[['system:index','user_id','weight_kg','longitude','latitude','RASTERVALU']]
data_map=data_map.rename(columns={'system:index':'SID','RASTERVALU':'YieldProxy'})

In [ ]:
df=data.drop(columns=['District_N', 'weight_kg', 'user_id', 'biomass_kg', 'latitude',
       'Ground_Tru', 'RASTERVALU', 'longitude'])
df=df.rename(columns={'system:index':'Index'})
df=df.set_index('Index')

In [ ]:
df=df.transpose()

In [ ]:
df['Rough']=df.index

In [ ]:
df['Date']=df['Rough'].apply(lambda x:x.split('_')[0].split('T')[0])

In [ ]:
df['Date']=pd.to_datetime(df['Date'],format='%Y%m%d')
df['Band']=df['Rough'].apply(lambda x:x.split('_')[-1])

In [ ]:
df=df.drop(columns=['Rough'])

In [ ]:
from datetime import datetime
g0 = '2022-11-10'
g1 = '2022-11-17'
g2 = '2023-01-11'
g3 = '2023-01-31'
g4 = '2023-03-07'
g0_date = datetime.strptime(g0,'%Y-%m-%d')
g1_date = datetime.strptime(g1,'%Y-%m-%d')
g2_date = datetime.strptime(g2,'%Y-%m-%d')
g3_date = datetime.strptime(g3,'%Y-%m-%d')
g4_date = datetime.strptime(g4,'%Y-%m-%d')

In [ ]:
def gsMap(t):
    cat=0
    if t<g1_date:
        cat=1
    elif t>=g1_date and t<g2_date:
        cat=2
    elif t>=g2_date and t<g3_date:
        cat=3
    elif t>=g3_date:
        cat=4
    return cat

In [ ]:
df['Growth Stage'] = df['Date'].apply(lambda x:gsMap(x))

In [ ]:
grp = df.groupby(['Band','Growth Stage']).mean()

In [ ]:
test=pd.DataFrame(grp)

In [ ]:
x=np.array(test.index)
a=[]
b=[]
for d in x:
    a.append(d[0])
    b.append(d[1])

In [ ]:
grp['BandName']=a
grp['grthstg']=b

In [ ]:
grp=grp.reset_index(drop=True)

In [ ]:
grp2=grp.groupby('grthstg')
vi_df=pd.DataFrame()
for d in grp2:
    ds=d[1]
    ds=ds.drop(columns=['grthstg'])
    ds=ds.set_index('BandName')
    ds=ds.transpose()
    ds['EVI']=2.5*((ds['B8']-ds['B4'])/(ds['B8']+6*ds['B4']-7.5*ds['B2']+1))
    ds['NDPI']= (ds['B8']-(0.74*ds['B4']+2.6*ds['B11']))/(ds['B8']+(0.74*ds['B4']+2.6*ds['B11']))
    ds['NDVI'] = (ds['B8']-ds['B4'])/(ds['B8']+ds['B4'])
    ds['SAVI'] = 1.5*(ds['B8']-ds['B4'])/(ds['B8']+ds['B4']+1.5)
    ds['NDWI'] = (ds['B8']-ds['B11'])/(ds['B8']+ds['B11'])
    ds['NDTI'] = (ds['B11']-ds['B12'])/(ds['B11']+ds['B12'])
    req = ds[['EVI', 'NDPI', 'NDVI', 'SAVI','NDWI', 'NDTI']]
    req=(req[req['SAVI']>0])
    req=req.add_suffix(str(d[0]))
    vi_df=pd.concat([vi_df,req],axis=1)

In [ ]:
vi_df['SID']=vi_df.index

In [ ]:
vi_df=vi_df[vi_df['NDVI1']>0]

In [ ]:
results=pd.merge(data_map,vi_df,on='SID')

In [ ]:
result=results[results['YieldProxy']>0]

In [ ]:
sns.heatmap(result.corr())

In [ ]:
from sklearn import preprocessing, svm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
result.columns

In [ ]:
X = np.array(result[['YieldProxy', 'EVI2',  'SAVI2',
        'EVI3', 'EVI4', 'NDPI4', 'NDVI4', 'SAVI4', 'NDWI4', 'NDTI4']])
y = np.array(result[['weight_kg']])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state=123)

In [ ]:
regr = LinearRegression()
regr.fit(X_train, y_train)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(n_estimators=1000,random_state=123, criterion='mse')

regressor.fit(X_train, y_train)

In [ ]:
X_data=np.array(results[['YieldProxy', 'EVI2',  'SAVI2',
        'EVI3', 'EVI4', 'NDPI4', 'NDVI4', 'SAVI4', 'NDWI4', 'NDTI4']])

In [ ]:
y_predcted_lr= regr.predict(X_data)
y_predcted_rf= regressor.predict(X_data)

In [ ]:
yld_lr=pd.DataFrame(y_predcted_lr)
yld_rf=pd.DataFrame(y_predcted_rf)

In [ ]:
results['Predicted(rf)']=yld_rf
results['Predicted(lr)']=yld_lr

In [ ]:
tst= results[(results['YieldProxy']>0) & (results['weight_kg']<15)]

In [ ]:
import seaborn as sns

In [ ]:
sns.scatterplot(data=tst,x='weight_kg',y='Predicted(rf)')

In [ ]:
def mape(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [ ]:
test=pd.read_excel(r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\bengalgram\gp shape\osmanabad\sentinel\Predicted_revised.xlsx")

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
y_test=np.array(test['weight_kg'])
y_pred=np.array(test['Predicted(rf)'])
mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
# squared True returns MSE value, False returns RMSE value.
mse = mean_squared_error(y_true=y_test,y_pred=y_pred) #default=True
rmse = mean_squared_error(y_true=y_test,y_pred=y_pred,squared=False)
r2 = r2_score(y_true=y_test,y_pred=y_pred)
mape = mape(y_test,y_pred)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R2:",r2)
print("R2:",r2)
print("MAPE:",mape)


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
tst.to_excel(r"C:\Users\pushkargaur\Desktop\miscellaneous\Neha\YieldData\bengalgram\gp shape\osmanabad\sentinel\Predicted_revised.xlsx")